In [1]:
#! Dataset download dependencies
%pip -q install datasets huggingface_hub

In [2]:
#! Imports and reproducibility
import hashlib
import io
import json
import os
import random
import shutil
import tarfile
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image as PILImage
from google.colab import drive
from sklearn.model_selection import train_test_split

SEED = 6304
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

drive.mount("/content/drive")

#! Existing project and shared Task 2/3 folders
ROOT = Path("/content/drive/MyDrive/pa1-beyond-iid")
PACS_DIR = ROOT / "data" / "pacs"
SPLIT_DIR = ROOT / "shared" / "splits"
SHARED_DIR = ROOT / "shared"
TASK2_DIR = ROOT / "task2"
TASK3_DIR = ROOT / "task3"

for folder in [PACS_DIR, SPLIT_DIR, TASK2_DIR / "configs", TASK2_DIR / "results", TASK2_DIR / "models", TASK3_DIR / "results"]:
    folder.mkdir(parents=True, exist_ok=True)

SOURCE_TAR = PACS_DIR / "pacs_sources.tar"
TARGET_TAR = PACS_DIR / "pacs_target_unlabeled.tar"
SOURCE_ALL = PACS_DIR / "source_all.csv"
SOURCE_TRAIN = PACS_DIR / "source_train.csv"
SOURCE_VAL = PACS_DIR / "source_val.csv"
TARGET_MANIFEST = PACS_DIR / "target_unlabeled.csv"
SPLIT_FILE = SPLIT_DIR / "pacs_sketch_seed6304.json"
PROVENANCE = PACS_DIR / "provenance.json"
LOCAL_ROOT = Path("/content/pacs_data")

SOURCE_DOMAINS = ["photo", "art_painting", "cartoon"]
TARGET_DOMAIN = "sketch"

DATA_READY = all(path.is_file() for path in [
    SOURCE_TAR, TARGET_TAR, SOURCE_ALL, SOURCE_TRAIN,
    SOURCE_VAL, TARGET_MANIFEST, SPLIT_FILE, PROVENANCE
])

print("Existing PACS setup:", "complete; reuse" if DATA_READY else "missing or incomplete; prepare")
print("Shared data:", PACS_DIR)

Mounted at /content/drive
Existing PACS setup: missing or incomplete; prepare
Shared data: /content/drive/MyDrive/pa1-beyond-iid/data/pacs


In [3]:
#! Download source data; keep target images label-free
from datasets import Image as HFImage, load_dataset
from huggingface_hub import HfApi

DATASET_REPO = "flwrlabs/pacs"

if not DATA_READY:
    DATASET_REVISION = HfApi().dataset_info(DATASET_REPO).sha
    hf_data = load_dataset(DATASET_REPO, split="train", revision=DATASET_REVISION)
    LABEL_NAMES = list(hf_data.features["label"].names)

    domains = hf_data["domain"]

    if set(domains) != set(SOURCE_DOMAINS + [TARGET_DOMAIN]):
        raise ValueError(f"Unexpected PACS domains: {set(domains)}")
    if len(LABEL_NAMES) != 7:
        raise ValueError(f"Expected seven classes, found {LABEL_NAMES}")

    hf_data = hf_data.cast_column("image", HFImage(decode=False))
    source_indices = [i for i, domain in enumerate(domains) if domain in SOURCE_DOMAINS]
    target_indices = [i for i, domain in enumerate(domains) if domain == TARGET_DOMAIN]

    #! Remove the target label column before accessing target examples.
    target_images_only = hf_data.select(target_indices).remove_columns("label")

    if "label" in target_images_only.column_names:
        raise RuntimeError("Target labels were not removed")

    print("Dataset revision:", DATASET_REVISION)
    print("Source images:", len(source_indices))
    print("Unlabeled target images:", len(target_indices))
    print("Classes:", LABEL_NAMES)
else:
    with open(PROVENANCE) as file:
        info = json.load(file)

    DATASET_REVISION = info["dataset_revision"]
    LABEL_NAMES = info["class_names"]
    print("Using saved dataset revision:", DATASET_REVISION)

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9991 [00:00<?, ? examples/s]

Dataset revision: 394113073258ead631f617d2e13bb377c0715c4b
Source images: 6062
Unlabeled target images: 3929
Classes: ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']


In [4]:
#! Export image bytes and prepare source/target manifests
FORMAT_EXT = {"JPEG": ".jpg", "PNG": ".png", "WEBP": ".webp", "BMP": ".bmp", "GIF": ".gif"}

def image_bytes(record):
    image = record["image"]
    blob = image.get("bytes")

    if blob is None:
        image_path = image.get("path")
        if image_path is None or not Path(image_path).is_file():
            raise FileNotFoundError("PACS image has neither bytes nor a readable path")
        blob = Path(image_path).read_bytes()

    with PILImage.open(io.BytesIO(blob)) as opened:
        fmt = opened.format
        opened.verify()

    if fmt not in FORMAT_EXT:
        raise ValueError(f"Unrecognized PACS image format: {fmt}")

    return blob, FORMAT_EXT[fmt]

def copy_to_drive_atomically(source, destination):
    temporary = destination.with_name(destination.name + ".partial")
    shutil.copy2(source, temporary)
    temporary.replace(destination)

if not DATA_READY:
    source_rows, target_rows = [], []

    with tempfile.TemporaryDirectory(prefix="pacs_export_", dir="/content") as scratch:
        stage = Path(scratch)

        #! Only source examples expose labels.
        for index in source_indices:
            row = hf_data[index]
            blob, extension = image_bytes(row)
            relative = Path("images") / row["domain"] / f"hf_{index:05d}{extension}"
            destination = stage / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(blob)

            source_rows.append({
                "hf_index": index, "image_id": f"hf_{index:05d}",
                "domain": row["domain"], "label": int(row["label"]),
                "label_name": LABEL_NAMES[int(row["label"])],
                "relative_path": relative.as_posix(),
                "sha256": hashlib.sha256(blob).hexdigest()
            })

        #! This view physically contains no target label field.
        for index, row in zip(target_indices, target_images_only):
            blob, extension = image_bytes(row)
            relative = Path("images") / TARGET_DOMAIN / f"hf_{index:05d}{extension}"
            destination = stage / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(blob)

            target_rows.append({
                "hf_index": index, "image_id": f"hf_{index:05d}",
                "domain": TARGET_DOMAIN, "relative_path": relative.as_posix(),
                "sha256": hashlib.sha256(blob).hexdigest()
            })

        #! Two archives let Task 3 extract only labeled source images.
        source_local = stage / "pacs_sources.tar"
        target_local = stage / "pacs_target_unlabeled.tar"

        with tarfile.open(source_local, "w") as archive:
            for domain in SOURCE_DOMAINS:
                archive.add(stage / "images" / domain, arcname=f"images/{domain}")

        with tarfile.open(target_local, "w") as archive:
            archive.add(stage / "images" / TARGET_DOMAIN, arcname=f"images/{TARGET_DOMAIN}")

        copy_to_drive_atomically(source_local, SOURCE_TAR)
        copy_to_drive_atomically(target_local, TARGET_TAR)

    pd.DataFrame(source_rows).sort_values("hf_index").to_csv(SOURCE_ALL, index=False)
    pd.DataFrame(target_rows).sort_values("hf_index").to_csv(TARGET_MANIFEST, index=False)

    print("Saved", len(source_rows), "labeled source images and", len(target_rows), "unlabeled target images")
else:
    print("Archives and manifests already present; skipping export.")

print("Source archive:", SOURCE_TAR)
print("Target archive:", TARGET_TAR)

Saved 6062 labeled source images and 3929 unlabeled target images
Source archive: /content/drive/MyDrive/pa1-beyond-iid/data/pacs/pacs_sources.tar
Target archive: /content/drive/MyDrive/pa1-beyond-iid/data/pacs/pacs_target_unlabeled.tar


In [5]:
#! Shared source train/validation split: reused unchanged by Task 3

if not DATA_READY:
    source = pd.read_csv(SOURCE_ALL)
    target = pd.read_csv(TARGET_MANIFEST)
    split_record = {
        "seed": SEED,
        "source_domains": SOURCE_DOMAINS,
        "target_domain": TARGET_DOMAIN,
        "source_splits": {},
        "target_unlabeled_indices": target["hf_index"].astype(int).tolist()
    }
    train_parts, val_parts = [], []

    for domain in SOURCE_DOMAINS:
        rows = source.loc[source["domain"] == domain].sort_values("hf_index")

        train_ids, val_ids = train_test_split(
            rows["hf_index"].to_numpy(), test_size=0.2,
            random_state=SEED, stratify=rows["label"].to_numpy()
        )
        train_ids, val_ids = sorted(train_ids.tolist()), sorted(val_ids.tolist())
        split_record["source_splits"][domain] = {
            "train_indices": train_ids, "val_indices": val_ids
        }

        train_parts.append(rows.set_index("hf_index").loc[train_ids].reset_index())
        val_parts.append(rows.set_index("hf_index").loc[val_ids].reset_index())

        print(domain, "train:", len(train_ids), "validation:", len(val_ids))

    train = pd.concat(train_parts, ignore_index=True)
    val = pd.concat(val_parts, ignore_index=True)
    train.to_csv(SOURCE_TRAIN, index=False)
    val.to_csv(SOURCE_VAL, index=False)

    with open(SPLIT_FILE, "w") as file:
        json.dump(split_record, file, indent=2)

    provenance = {
        "dataset_repo": DATASET_REPO,
        "dataset_revision": DATASET_REVISION,
        "class_names": LABEL_NAMES,
        "source_domains": SOURCE_DOMAINS,
        "target_domain": TARGET_DOMAIN,
        "source_images": len(source),
        "target_images": len(target),
        "source_archive_sha256": hashlib.sha256(SOURCE_TAR.read_bytes()).hexdigest(),
        "target_archive_sha256": hashlib.sha256(TARGET_TAR.read_bytes()).hexdigest(),
        "target_labels_exported": False,
        "split_seed": SEED
    }

    with open(PROVENANCE, "w") as file:
        json.dump(provenance, file, indent=2)

else:
    train = pd.read_csv(SOURCE_TRAIN)
    val = pd.read_csv(SOURCE_VAL)
    target = pd.read_csv(TARGET_MANIFEST)

    with open(SPLIT_FILE) as file:
        split_record = json.load(file)

    print("Reusing saved source splits.")

#! Minimal split-integrity tests
if set(train["hf_index"]) & set(val["hf_index"]):
    raise ValueError("Source train/validation overlap")

if "label" in target.columns or "label_name" in target.columns:
    raise ValueError("Target manifest unexpectedly exposes labels")

if len(train) + len(val) != len(pd.read_csv(SOURCE_ALL)):
    raise ValueError("Source split is incomplete")

print("Source train:", len(train), "Source validation:", len(val))
print("Target image-only manifest:", len(target))
print("Splits saved:", SPLIT_FILE)

photo train: 1336 validation: 334
art_painting train: 1638 validation: 410
cartoon train: 1875 validation: 469
Source train: 4849 Source validation: 1213
Target image-only manifest: 3929
Splits saved: /content/drive/MyDrive/pa1-beyond-iid/shared/splits/pacs_sketch_seed6304.json


In [6]:
#! Persist shared loading, transforms, model and BatchNorm policy
PACS_MODULE = r'''#! Shared PACS protocol for Tasks 2 and 3

import random
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 6304
SOURCE_DOMAINS = ("photo", "art_painting", "cartoon")
TARGET_DOMAIN = "sketch"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _extract_safe(archive_path, destination):
    root = Path(destination).resolve()

    with tarfile.open(archive_path, "r") as archive:
        for member in archive.getmembers():
            output = (root / member.name).resolve()

            if not output.is_relative_to(root) or member.issym() or member.islnk():
                raise ValueError(f"Unsafe archive member: {member.name}")

        archive.extractall(root)


def prepare_local_data(storage_root, local_root, include_target=False):
    storage_root, local_root = Path(storage_root), Path(local_root)
    local_root.mkdir(parents=True, exist_ok=True)
    source_file = local_root / "images" / "photo"

    if not source_file.is_dir() or not any(source_file.iterdir()):
        _extract_safe(storage_root / "pacs_sources.tar", local_root)

    if include_target:
        target_file = local_root / "images" / TARGET_DOMAIN
        if not target_file.is_dir() or not any(target_file.iterdir()):
            _extract_safe(storage_root / "pacs_target_unlabeled.tar", local_root)
    return local_root


def make_transform(train=False):
    stats = models.ResNet18_Weights.IMAGENET1K_V1.transforms()
    geometry = [transforms.Resize((256, 256))]

    if train:
        geometry += [transforms.RandomCrop(224), transforms.RandomHorizontalFlip()]
    else:
        geometry += [transforms.CenterCrop(224)]
    return transforms.Compose(geometry + [
        transforms.ToTensor(), transforms.Normalize(mean=stats.mean, std=stats.std)
    ])


class PACSDataset(Dataset):
    def __init__(self, manifest, image_root, transform, domain=None, unlabeled=False):
        self.records = pd.read_csv(manifest)

        if domain is not None:
            self.records = self.records[self.records["domain"] == domain]

        self.records = self.records.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.transform = transform
        self.unlabeled = unlabeled

        if unlabeled and ("label" in self.records or "label_name" in self.records):
            raise ValueError("Unlabeled target manifest contains labels")

        if not unlabeled and "label" not in self.records:
            raise ValueError("Source manifest is missing labels")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        path = self.image_root / row["relative_path"]

        with Image.open(path) as image:
            rgb = image.convert("RGB")
        tensor = self.transform(rgb)

        if self.unlabeled:
            return tensor

        return tensor, int(row["label"])


def make_loaders(storage_root, local_root, include_target=False, workers=2):
    storage_root = Path(storage_root)
    image_root = prepare_local_data(storage_root, local_root, include_target=include_target)
    source = {}
    validation = {}

    for i, domain in enumerate(SOURCE_DOMAINS):
        train_ds = PACSDataset(storage_root / "source_train.csv", image_root, make_transform(True), domain=domain)
        val_ds = PACSDataset(storage_root / "source_val.csv", image_root, make_transform(False), domain=domain)
        gen = torch.Generator().manual_seed(SEED + i)

        source[domain] = DataLoader(train_ds, batch_size=8, shuffle=True, drop_last=True,
                                    num_workers=workers, pin_memory=torch.cuda.is_available(), generator=gen)
        validation[domain] = DataLoader(val_ds, batch_size=64, shuffle=False,
                                        num_workers=workers, pin_memory=torch.cuda.is_available())

    if not include_target:
        return source, validation

    target_ds = PACSDataset(storage_root / "target_unlabeled.csv", image_root,
                            make_transform(True), unlabeled=True)

    target_gen = torch.Generator().manual_seed(SEED + 100)
    target_loader = DataLoader(target_ds, batch_size=24, shuffle=True, drop_last=True,
                               num_workers=workers, pin_memory=torch.cuda.is_available(), generator=target_gen)

    return source, validation, target_loader


def forever(loader):
    while True:
        yield from loader


def build_resnet18(num_classes=7):
    backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    backbone.fc = nn.Linear(backbone.fc.in_features, num_classes)
    return backbone


def freeze_bn_running_stats(model):
    #! Call AFTER model.train(); preserve trainable BatchNorm affine parameters.
    for layer in model.modules():
        if isinstance(layer, nn.modules.batchnorm._BatchNorm):
            layer.eval()
'''

SHARED_FILE = SHARED_DIR / "pacs.py"
SHARED_FILE.write_text(PACS_MODULE, encoding="utf-8")
print("Reusable Python module saved:", SHARED_FILE)

Reusable Python module saved: /content/drive/MyDrive/pa1-beyond-iid/shared/pacs.py


In [7]:
#! Test balanced source and unlabeled target batches
import sys
sys.path.insert(0, str(SHARED_DIR))

from pacs import make_loaders, freeze_bn_running_stats

source_loaders, val_loaders, target_loader = make_loaders(
    PACS_DIR, LOCAL_ROOT, include_target=True, workers=2
)

for domain, loader in source_loaders.items():
    images, labels = next(iter(loader))

    if images.shape != (8, 3, 224, 224) or labels.shape != (8,):
        raise ValueError(f"Unexpected {domain} training batch")

    print(domain, "source:", tuple(images.shape), "labels:", tuple(labels.shape),
          "validation images:", len(val_loaders[domain].dataset))

target_images = next(iter(target_loader))

if not isinstance(target_images, torch.Tensor) or target_images.shape != (24, 3, 224, 224):
    raise ValueError("Target loader must return 24 images and NO labels")

print("Sketch unlabeled batch:", tuple(target_images.shape))
print("No target labels are present in target loader or target manifest.")

photo source: (8, 3, 224, 224) labels: (8,) validation images: 334
art_painting source: (8, 3, 224, 224) labels: (8,) validation images: 410
cartoon source: (8, 3, 224, 224) labels: (8,) validation images: 469
Sketch unlabeled batch: (24, 3, 224, 224)
No target labels are present in target loader or target manifest.


In [8]:
#! Save locked experimental protocol before target-label evaluation
BASE_CONFIG = {
    "seed": SEED,
    "dataset_repo": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "class_names": LABEL_NAMES,
    "source_domains": SOURCE_DOMAINS,
    "target_domain": TARGET_DOMAIN,
    "source_train_manifest": str(SOURCE_TRAIN),
    "source_validation_manifest": str(SOURCE_VAL),
    "target_unlabeled_manifest_task2_only": str(TARGET_MANIFEST),
    "split_file": str(SPLIT_FILE),
    "network": "torchvision resnet18; ResNet18_Weights.IMAGENET1K_V1; 7-class fc",
    "fine_tune_backbone": True,
    "train_transform": "Resize((256,256)), RandomCrop(224), RandomHorizontalFlip(), ImageNet Normalize",
    "eval_transform": "Resize((256,256)), CenterCrop(224), ImageNet Normalize",
    "batch_size_per_source_domain": 8,
    "target_batch_size_task2_only": 24,
    "optimizer": "AdamW",
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "max_source_epochs": 30,
    "early_stopping_patience": 5,
    "checkpoint_selection": "mean_macro_f1_across_photo_art_painting_cartoon_validation",
    "batchnorm_running_stats": "ImageNet pretrained; frozen by BN.eval() after model.train()",
    "batchnorm_affine": "trainable",
    "dan_main_lambda": 1.0,
    "dan_kernel_multipliers": [0.5, 1.0, 2.0],
    "dan_controlled_study_lambdas": [0.1, 1.0, 10.0],
    "target_labels": "final evaluation ONLY, after all settings/checkpoints fixed",
    "task3": "sources-only archive; no Sketch images, target labels or target results in training/selection"
}

CONFIG_PATH = TASK2_DIR / "configs" / "base.json"

if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as file:
        previous = json.load(file)

    if previous != BASE_CONFIG:
        raise RuntimeError("Existing frozen base config differs. Do not overwrite experimental decisions.")

    print("Existing config preserved.")
else:
    with open(CONFIG_PATH, "w") as file:
        json.dump(BASE_CONFIG, file, indent=2)
    print("Frozen base configuration saved.")

required = [SOURCE_TAR, TARGET_TAR, SOURCE_TRAIN, SOURCE_VAL, TARGET_MANIFEST,
            SPLIT_FILE, PROVENANCE, SHARED_FILE, CONFIG_PATH]

missing = [str(path) for path in required if not path.is_file()]

if missing:
    raise FileNotFoundError(f"Missing setup artifacts: {missing}")

print("TASK 2A COMPLETE")
print("Artifacts verified:", len(required))
print("Next notebook: Task 2B — Source-only ERM (T4 recommended)")

Frozen base configuration saved.
TASK 2A COMPLETE
Artifacts verified: 9
Next notebook: Task 2B — Source-only ERM (T4 recommended)
